## Imports

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from dotenv import load_dotenv
import sqlite3
import requests


## State Class

In [2]:
class ChatState(TypedDict):

    title: str
    messages: Annotated[list[BaseMessage], add_messages]

## Tools 

In [3]:
@tool
def calculator(num1: float, num2: float, operation: str) -> dict:
    """
    A simple calculator function that performs basic arithmetic operations.
    
    Parameters:
    -----------
    num1 : float
        The first number for the calculation
    num2 : float
        The second number for the calculation
    operation : str
        The arithmetic operation to perform. Valid options:
        - 'add' or '+': Addition
        - 'subtract' or '-': Subtraction
        - 'multiply' or '*': Multiplication
        - 'divide' or '/': Division
        - 'power' or '**': Exponentiation
        - 'modulus' or '%': Modulus (remainder)
        - 'floor_divide' or '//': Floor division
    
    Returns:
    --------
    float or str
        The result of the calculation, or an error message if invalid
    
    Examples:
    ---------
    >>> calculator(10, 5, 'add')
    15.0
    >>> calculator(10, 5, '/')
    2.0
    >>> calculator(2, 3, 'power')
    8.0
    """
    
    # Convert inputs to float to handle both integers and decimals
    try:
        num1 = float(num1)
        num2 = float(num2)
    except (TypeError, ValueError):
        return {"error": "Invalid number input. Please provide numeric values."}
    
    # Convert operation to lowercase for case-insensitive matching
    operation = str(operation).lower().strip()
    
    # Perform the requested operation
    if operation in ['add', '+']:
        # Addition: num1 + num2
        result = num1 + num2
        
    elif operation in ['subtract', '-']:
        # Subtraction: num1 - num2
        result = num1 - num2
        
    elif operation in ['multiply', '*']:
        # Multiplication: num1 * num2
        result = num1 * num2
        
    elif operation in ['divide', '/']:
        # Division: num1 / num2
        # Check for division by zero
        if num2 == 0:
            return {"Error": "Division by zero is undefined."}
        result = num1 / num2
        
    elif operation in ['power', '**']:
        # Exponentiation: num1 raised to the power of num2
        try:
            result = num1 ** num2
        except OverflowError:
            return {"Error": "Result too large to compute."}
            
    elif operation in ['modulus', '%']:
        # Modulus: remainder of num1 divided by num2
        # Check for modulus by zero
        if num2 == 0:
            return {"Error": "Modulus by zero is undefined."}
        result = num1 % num2
        
    elif operation in ['floor_divide', '//']:
        # Floor division: integer division of num1 by num2
        # Check for division by zero
        if num2 == 0:
            return {"Error": "Floor division by zero is undefined."}
        result = num1 // num2
        
    else:
        # Invalid operation provided
        return {"Error": (f"Invalid operation '{operation}'. "
                "Valid operations: add(+), subtract(-), multiply(*), "
                "divide(/), power(**), modulus(%), floor_divide(//)")}
    
    # Return the calculated result
    return {"first_num": num1, "second_num": num2, "operation": operation, "result": result}

In [4]:
tools = [calculator]

## LLM

In [5]:
llm = ChatOpenAI(model="gpt-5-mini")

llm_with_tools = llm.bind_tools(tools)

## Node functions

In [6]:
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

In [36]:
def generate_chat_title(state: ChatState):
    """Node that will generate a title for the node"""

    if ('title' not in state) or state['title'] or state['title'] == '':
        prompt = f"Generate a short title based on the {state['messages']}"
        response = llm.invoke(prompt)
        return {"title": [response]}



## Create checkpointer

In [8]:
conn = sqlite3.connect(database="chatbot.db", check_same_thread=False)
checkpointer = SqliteSaver(conn=conn)

## Create Graph

In [37]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node('chat_title_node', generate_chat_title)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")
graph.add_edge(START, "chat_title_node")
graph.add_conditional_edges("chat_node",tools_condition)
graph.add_edge('tools', 'chat_node')
graph.add_edge('chat_title_node', END)

chatbot = graph.compile() # checkpointer=checkpointer

In [38]:
chatbot.get_graph().print_ascii()

            +-----------+                 
            | __start__ |                 
            +-----------+                 
            ***         ***               
           *               *              
         **                 **            
+-----------+         +-----------------+ 
| chat_node |.        | chat_title_node | 
+-----------+ ...     +-----------------+ 
      .          .....          *         
      .               ...       *         
      .                  ...    *         
  +-------+               +---------+     
  | tools |               | __end__ |     
  +-------+               +---------+     


In [39]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of india')]
}

chatbot.invoke(initial_state)['title']

[AIMessage(content='Capital of India', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 58, 'total_tokens': 198, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CLTV6vVmnm3a7EOeGyFnjfkEqkYBw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--5b93e229-c43d-4afc-995e-98e3fd6e6635-0', usage_metadata={'input_tokens': 58, 'output_tokens': 140, 'total_tokens': 198, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}})]